In [14]:
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import RandomizedSearchCV, train_test_split
from sklearn.inspection import permutation_importance
from scipy.stats import randint
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np
import joblib


In [15]:
def clean_fare(value: str) -> float:
    return float(value.replace("$", ""))

In [16]:
url = "https://raw.githubusercontent.com/Edwz208/AirfareTicketAnalysis/8a7b623eb19e68c7612d39a811f1d6cd1d2d4eef/airline_ticket_dataset.csv"

df = pd.read_csv(url)

columns = [
    "Year", "quarter", "citymarketid_1", "citymarketid_2",
    "city1", "city2", "nsmiles", "passengers", "fare",
    "carrier_lg", "large_ms", "fare_lg", "carrier_low",
    "lf_ms", "fare_low",
    "TotalFaredPax_city1", "TotalPerLFMkts_city1", "TotalPerPrem_city1",
    "TotalFaredPax_city2", "TotalPerLFMkts_city2", "TotalPerPrem_city2"
]

df = df[columns]

# Clean money columns
df["fare_lg"] = df["fare_lg"].apply(clean_fare)
df["fare_low"] = df["fare_low"].apply(clean_fare)
df["fare"] = df["fare"].apply(clean_fare)

# Ensure numeric conversion of passengers, currently strings
df["passengers"] = pd.to_numeric(df["passengers"], errors="coerce")

X = df.drop(columns=["fare", "fare_low", "fare_lg", "citymarketid_1", "citymarketid_2"])
y = df["fare"]

# One-hot encoding
cat_cols = ["city1", "city2", "carrier_lg", "carrier_low"]
num_cols = [c for c in X.columns if c not in cat_cols]

In [17]:
preprocess = ColumnTransformer(
    transformers=[
        ("num", SimpleImputer(strategy="median"), num_cols),
        ("cat", Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore"))
        ]), cat_cols),
    ],
    remainder="drop"
)

model = RandomForestRegressor(
    bootstrap=True,
    max_depth=24,
    min_samples_leaf=3,
    min_samples_split=2,
    n_estimators=100,
    max_features=1/3,       # fraction of features chosen per split
    oob_score=True,         # enable out-of-bag evaluation
    random_state=0
)

pipe = Pipeline(steps=[
    ("preprocess", preprocess),
    ("model", model)
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, random_state=0
)

pipe.fit(X_train, y_train)

print(f"Baseline OOB score: {pipe.named_steps['model'].oob_score_:.8f}")
print("Test R^2:", pipe.score(X_test, y_test))

Baseline OOB score: 0.88926488
Test R^2: 0.8841663883783317


In [ ]:
# Parameter restrictions for tuning
param_dist = {
    "model__max_features": ["sqrt", "log2", 1/3, 0.3, 0.5, None],
    "model__max_depth": [None, 8, 12, 16, 24],
    "model__min_samples_split": randint(2, 11),
    "model__min_samples_leaf": randint(1, 6),
    "model__bootstrap": [True],
}

search = RandomizedSearchCV(
    pipe,
    param_distributions=param_dist,
    n_iter=80,
    cv=5,
    scoring="r2",
    n_jobs=-1,
    random_state=0,
    verbose=1
)

# trains many pipelines and picks best
search.fit(X_train, y_train)

pipe = search.best_estimator_

print("Best params:", search.best_params_)
print("Best CV R^2:", round(search.best_score_, 3))
print("Test R^2:", pipe.score(X_test, y_test))

print("OOB:", pipe.named_steps["model"].oob_score_)

Fitting 5 folds for each of 80 candidates, totalling 400 fits


In [ ]:
feature_names = pipe.named_steps["preprocess"].get_feature_names_out()

perm_importance = permutation_importance(pipe, X_test, y_test, n_repeats=10, random_state=0)

sorted_idx = perm_importance.importances_mean.argsort()[::-1]

for i in sorted_idx:

    print(f"{X.columns[i]}: {perm_importance.importances_mean[i]:.8f}")

In [ ]:
def predict_fare(pipe, **kwargs) -> float:
    sample = pd.DataFrame([kwargs], columns=X.columns)
    pred = pipe.predict(sample)[0]
    return float(pred)

In [ ]:
preds = pipe.predict(X_test)

print("MAE:", mean_absolute_error(y_test, preds))
print("RMSE:", np.sqrt(mean_squared_error(y_test, preds)))

for i in range(50):
    sample = X.iloc[[i]]
    prediction = pipe.predict(sample)[0]
    print(f"{X.index[i]}: ${prediction:,.2f}")

In [ ]:
joblib.dump(pipe, "airfare_model.pkl")

print("Model saved successfully.")